In [14]:
!python settings.py

Using device: cuda


In [15]:
import os
import pandas as pd
from ast import literal_eval
from tqdm.autonotebook import tqdm

from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    SentenceTransformerModelCardData,
)
from sentence_transformers.readers       import InputExample
from sentence_transformers.models        import Transformer, Pooling
from sentence_transformers.losses        import CachedMultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers

from settings import MODEL_ID, MODEL_NAME, LOG_DIR, CACHE_DIR, OUTPUT_DIR, MAX_SEQ_LEN, EPOCHS, LR, BATCH_SIZE, DEVICE

os.environ['WANDB_DISABLED'] = 'true'

In [16]:
data = {
    'corpus': pd.read_parquet('data/processed/corpus_data.parquet'),
    'train' : pd.read_parquet('data/processed/train_data.parquet'),
    'test'  : pd.read_parquet('data/processed/test_data.parquet')
}
for split in ['train', 'test']:
    data[split]['cid']          = data[split]['cid'].apply(lambda x: x.tolist())
    data[split]['context_list'] = data[split]['context_list'].apply(lambda x: x.tolist())
    
examples = {'train': [], 'test': []}

In [17]:
data['train'].head()

,question,context_list,qid,cid
0,Liên đoàn Luật sư Việt Nam là tổ chức xã hội –...,[“Điều 2. Địa vị pháp lý của Liên đoàn Luật sư...,72600,[142820]
1,Tên hợp tác xã bị rơi vào trường hợp cấm thì c...,"[""Điều 7. Tên hợp tác xã, liên hiệp hợp tác xã...",147562,"[27817, 72117]"
2,Tài xế lái xe ô tô khách 50 chỗ ngồi bao lâu t...,"[""1. Sử dụng lái xe bảo đảm sức khỏe theo tiêu...",142107,"[33215, 56201]"
3,Các bước chuẩn bị thủ thuật bó bột Cravate sẽ ...,[BỘT CRAVATE\n...\nIV. CHUẨN BỊ\n1. Người thực...,77353,[148158]
4,Viên chức Hộ sinh hạng 4 có những nhiệm vụ gì ...,[Hộ sinh hạng IV - Mã số: V.08.06.16\n1. Nhiệm...,113090,[188132]


In [18]:
# Debug
for col in data['test'].columns:
    print(col, type(data['test'][col][0]))
    
print((data['test']['cid'].apply(len) == data['test']['context_list'].apply(len)).all())

question <class 'str'>
context_list <class 'list'>
qid <class 'numpy.int64'>
cid <class 'list'>
True


In [19]:
for split in ['train', 'test']:
    rows = list(data[split].itertuples(index=False))
    
    for row in tqdm(rows, desc=f"Processing {split}", unit='rows'):
        q = row.question
        for c in row.context_list:
            examples[split].append(InputExample(texts=[q, c]))

print(f"Training examples: {len(examples['train'])}") # Compare with sum(data['train']['cid'].apply(len))

Processing test: 100%|██████████| 29723/29723 [00:00<00:00, 640686.27rows/s]

Training examples: 99580


In [ ]:
embedding_model = Transformer(MODEL_ID, max_seq_length=MAX_SEQ_LEN, cache_dir=CACHE_DIR)
pooling_model   = Pooling(
    embedding_model.get_word_embedding_dimension(), 
    pooling_mode_mean_tokens=True
)

model = SentenceTransformer(
    modules=[embedding_model, pooling_model], device=DEVICE, 
    cache_folder=CACHE_DIR,
    model_card_data=SentenceTransformerModelCardData(
        model_id=MODEL_ID, 
        model_name=MODEL_NAME, 
        model_description='Fine-tuned Sentence-BERT model for Vietnamese legal documents retrieval',
        language='vi',
        license='MIT',
    )
)

In [ ]:
loss = CachedMultipleNegativesRankingLoss(model=model)

args = SentenceTransformerTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LR,
    warmup_ratio=0.1,
    fp16=True,
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    logging_dir=LOG_DIR,
    logging_steps=100
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=examples['train'],
    loss=loss
)
trainer.train()

In [ ]:
model.save_pretrained(OUTPUT_DIR)
# model.push_to_hub(
#     repo_id=model_name, 
#     commit_message='Upload model to Hugging Face Hub',
#     private=True,
#     use_auth_token=True
# )